## DEMO

In [1]:
!pip install -r requirements.txt

  Using cached sentence_transformers-4.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached regex-2024.11.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached MarkupSafe-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (4.0 kB)
Using cached sentence_transformers-4.1.0-py3-none-any.whl (345 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
Using cached regex-2024.11.6-cp312-cp312-macosx_11_0_arm64.whl (284 kB)
Using cached safetensors-0.5.3-cp38-abi3-macosx_11_0_arm64.whl (418 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 11.7 MB/s eta 0:00:00a 0:00:01
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached MarkupSafe-3.0.2-cp312-cp312-macosx_11_0_arm64.whl (12 

### Setup

In [1]:

from agent.orchestrator_agent import OrchestratorAgent
import chromadb
from dotenv import load_dotenv
import json
import os
from db.chroma_functions import *
import random


# Load environment variables
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")

#Initialize ChromaDB  
chroma_client = chromadb.PersistentClient(path="./data/chroma_db")

cves = json.load(open('data/simulated_cves.json'))
collection = chroma_setup(cves)
orchestrator = OrchestratorAgent(
    gemini_api_key=gemini_api_key, 
    chroma_client=chroma_client, 
)

# Load incident data
incidents_json = json.load(open('data/incidents.json'))

random_index = random.randint(0, 4)
incident_data = incidents_json[random_index]

Setting up
Creating client with persistence


/Users/206801024/virtual_envrionments/python_3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection 'cves' already exists. Retrieving it.
Collection 'cves' retrieved successfully.


### Run

In [2]:
incident_data

{'incident_id': 'INC-2023-08-01-003',
 'timestamp': '2023-08-01T14:30:00Z',
 'title': 'Web Application SQL Injection Attempt',
 'description': 'Web application firewall blocked multiple requests containing suspected SQL injection payloads targeting the login page.',
 'affected_assets': [{'hostname': 'web-app-server-05',
   'ip_address': '10.10.5.20',
   'os': 'CentOS 7',
   'installed_software': [{'name': 'Apache Tomcat', 'version': '9.0.50'},
    {'name': 'MySQL Connector/J', 'version': '8.0.25'}],
   'role': 'Internal Web Application Server'}],
 'observed_ttps': [{'framework': 'MITRE ATT&CK',
   'id': 'T1190',
   'name': 'Exploit Public-Facing Application'},
  {'framework': 'MITRE ATT&CK',
   'id': 'T1059.004',
   'name': 'Command and Scripting Interpreter: SQL'}],
 'indicators_of_compromise': [{'type': 'ip_address',
   'value': '52.1.2.3',
   'context': 'Source IP of attack attempts'},
  {'type': 'uri_path', 'value': '/login.jsp', 'context': 'Targeted page'}],
 'initial_findings': '

In [3]:
orchestrator_result = orchestrator.call_orchestrator_agent(incident_data=incident_data)


### Result

In [28]:
print(orchestrator.threat_analysis)

Based on the provided incident report and CVE data, here's an analysis:

**Incident Summary:**

The incident involved a suspected SQL injection attempt targeting the login page of an internal web application ("web-app-server-05") running Apache Tomcat 9.0.50 and using a MySQL Connector/J 8.0.25.  The attack originated from IP address 52.1.2.3.  The web application firewall successfully blocked the malicious requests.

**CVE Prioritization and Relevance:**

The following CVEs are prioritized based on their relevance to the incident details, considering the affected software and the type of attack:


1. **CVE-2023-10083 (MySQL 8.0.26 privilege escalation via crafted SQL query):** This is the *most* relevant CVE. The incident involved a SQL injection attempt against a MySQL database accessed via the MySQL Connector/J library. While the version in use (8.0.25) is slightly older than the one mentioned in the CVE (8.0.26),  privilege escalation vulnerabilities in MySQL are highly relevant to